# 06  Contracts Finder signals (first SME-reaching source)

NB05 built the spine and wired up the exact id bridges (GLEIF, Wikidata). Those only reach the
few larger firms, since SMEs rarely hold an LEI or trade publicly. This notebook adds the first
signal that genuinely reaches small firms: **public-sector contract wins** from Contracts Finder.

A company winning a government contract is a useful commercial signal (revenue, growth, a reason
the bank might engage). Contracts Finder is free, needs no key, and many suppliers are SMEs.

The supplier on a contract is a name, not a company number, so this is where the **matching
ladder** earns its keep:
1. exact Companies House number when the notice carries one (confidence 1.0),
2. exact normalised name when it does not (confidence 0.9),
3. RapidFuzz best match within a first-word block, above a score cutoff (confidence scaled).

Every match it writes into the `signals` table carries its source and confidence, so a clean id
join is always distinguishable from a fuzzy name guess.

## How to run
Same as NB05. In Colab via Drive: this reads `lloyds.duckdb` from `MyDrive/Lloyds` (built by
NB05), so run NB05 first. Then run this notebook top to bottom; it updates the same database.


## 1. Install and import

In [ ]:
import sys, subprocess
for pkg in ["duckdb", "rapidfuzz"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

In [ ]:
import duckdb
import pandas as pd
import requests
import re, time
from collections import defaultdict
from datetime import datetime, timezone, timedelta
from rapidfuzz import process, fuzz

print("duckdb", duckdb.__version__, "| pandas", pd.__version__)

## 2. Locate the database
Find the `lloyds.duckdb` that NB05 built. In Colab we read it from the same Drive folder; locally
from `data/processed`. The database must already exist (run NB05 first).

`LOOKBACK_DAYS` sets how far back to pull awarded contracts. `CF_MAX_PAGES` caps the pull (100
notices per page); set it to `None` to pull the whole window.

In [ ]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/Lloyds")   # <- same folder as NB05
    DB_PATH = WORK_DIR / "lloyds.duckdb"
else:
    DB_PATH = Path("..").resolve() / "data" / "processed" / "lloyds.duckdb"

assert DB_PATH.exists(), f"lloyds.duckdb not found at {DB_PATH}. Run NB05 first."
print("DB:", DB_PATH)

LOOKBACK_DAYS = 365     # how far back to pull awarded contracts
CF_MAX_PAGES  = 30      # 100 notices/page; set to None for the whole window

USER_AGENT = "LloydsBCB-MSc-project/1.0 (academic; contact via GitHub elyokerr)"
NOW = datetime.now(timezone.utc).isoformat(timespec="seconds")

## 3. Helpers
The same company-number and name cleaners as NB05, so matching uses identical normalisation.

In [ ]:
def clean_company_number(value):
    if value is None:
        return None
    s = str(value).strip().upper()
    if s == "" or s == "NAN":
        return None
    if s.isdigit():
        return s.zfill(8)
    return s

_SUFFIXES = [
    "LIMITED", "LTD", "PLC", "PUBLIC LIMITED COMPANY", "LLP",
    "LIMITED LIABILITY PARTNERSHIP", "LP", "CIC", "CIO",
    "COMPANY", "CO", "AND", "THE",
]
_SUFFIX_RE = re.compile(r"\b(" + "|".join(_SUFFIXES) + r")\b")

def normalise_name(name):
    if name is None:
        return None
    s = str(name).upper()
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    s = _SUFFIX_RE.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s or None

print("helpers ok")

## 4. Load the spine and build the match indexes
We pull the company number, normalised name, and postcode for every company, then build two
lookups: an exact name index, and first-word blocks so the fuzzy step only compares names that
start with the same word (this keeps it fast even on the full 1.37M dataset).

In [ ]:
con = duckdb.connect(str(DB_PATH))
spine = con.execute("SELECT company_number, name_norm, postcode FROM companies").df()
print(f"spine: {len(spine):,} companies")

spine_numbers = set(spine["company_number"])
by_name = defaultdict(list)     # name_norm -> [company_number, ...]
blocks  = defaultdict(list)     # first word -> [(name_norm, company_number), ...]
for cn, nm in zip(spine["company_number"], spine["name_norm"]):
    if not nm:
        continue
    by_name[nm].append(cn)
    blocks[nm.split(" ")[0]].append((nm, cn))
print(f"exact-name keys: {len(by_name):,} | first-word blocks: {len(blocks):,}")

## 5. The matching ladder
Given a supplier name and an optional Companies House number, return the best company number in
the spine, plus a confidence and the method used. `FUZZY_CUTOFF` is the minimum RapidFuzz score
(0-100) we accept for a fuzzy match.

In [ ]:
FUZZY_CUTOFF = 90

def match_supplier(name, ch_number):
    # 1. exact Companies House number on the notice
    cn = clean_company_number(ch_number)
    if cn and cn in spine_numbers:
        return cn, 1.0, "company_number"

    nm = normalise_name(name)
    if not nm:
        return None, 0.0, "no_match"

    # 2. exact normalised name
    if nm in by_name:
        cands = by_name[nm]
        if len(cands) == 1:
            return cands[0], 0.9, "name_exact"
        return cands[0], 0.5, "name_exact_ambiguous"   # several firms share the name

    # 3. fuzzy within the first-word block
    bucket = blocks.get(nm.split(" ")[0])
    if bucket:
        choices = [b[0] for b in bucket]
        hit = process.extractOne(nm, choices, scorer=fuzz.WRatio, score_cutoff=FUZZY_CUTOFF)
        if hit:
            _, score, idx = hit
            return bucket[idx][1], round(score / 100 * 0.85, 3), "name_fuzzy"

    return None, 0.0, "no_match"

# sanity check on a name we know is in the sample
print(match_supplier("AAR SERVICES LIMITED", None))

## 6. Pull awarded contracts from Contracts Finder
Keyless OCDS search, filtered to awarded notices over the lookback window, following the cursor
until the window is exhausted or `CF_MAX_PAGES` is hit. The API asks callers to back off for five
minutes after a 403, so we stop politely if we hit one. Results are cached so re-runs are quick.

In [ ]:
CF_URL = "https://www.contractsfinder.service.gov.uk/Published/Notices/OCDS/Search"

def fetch_contracts(lookback_days=LOOKBACK_DAYS, max_pages=CF_MAX_PAGES):
    cache = DB_PATH.parent / f"cache_contracts_{lookback_days}d.parquet"
    if cache.exists():
        print("  using cached Contracts Finder pull")
        return pd.read_parquet(cache)

    published_to   = datetime.now(timezone.utc)
    published_from = published_to - timedelta(days=lookback_days)
    params = {
        "stages": "award",
        "publishedFrom": published_from.strftime("%Y-%m-%dT%H:%M:%S"),
        "publishedTo":   published_to.strftime("%Y-%m-%dT%H:%M:%S"),
        "limit": 100,
    }
    releases, url, first, page = [], CF_URL, True, 0
    while True:
        r = requests.get(url, params=params if first else None,
                         headers={"User-Agent": USER_AGENT}, timeout=60)
        if r.status_code in (403, 429):
            print(f"  rate limited at page {page}, stopping (re-run later to continue)")
            break
        r.raise_for_status()
        body = r.json()
        releases.extend(body.get("releases", []))
        page += 1
        nxt = body.get("links", {}).get("next")
        if not nxt or (max_pages is not None and page >= max_pages):
            break
        url, first = nxt, False
        time.sleep(0.3)   # be polite

    df = _extract_awards(releases)
    df.to_parquet(cache, index=False)
    return df

def _extract_awards(releases):
    rows = []
    for rel in releases:
        # map a party id to its Companies House number where the notice gives one
        party_ch = {}
        for p in rel.get("parties", []) or []:
            pid = p.get("id")
            for ident in [p.get("identifier")] + (p.get("additionalIdentifiers") or []):
                if ident and "COH" in str(ident.get("scheme", "")).upper():
                    party_ch[pid] = ident.get("id")
        for aw in rel.get("awards", []) or []:
            date  = aw.get("date") or (aw.get("contractPeriod") or {}).get("startDate")
            value = (aw.get("value") or {}).get("amount")
            title = aw.get("title") or (rel.get("tender") or {}).get("title")
            for sup in aw.get("suppliers", []) or []:
                rows.append({
                    "supplier_name": sup.get("name"),
                    "ch_number": party_ch.get(sup.get("id")),
                    "award_date": date,
                    "value": value,
                    "detail": title,
                })
    return pd.DataFrame(rows, columns=["supplier_name", "ch_number", "award_date", "value", "detail"])

awards = fetch_contracts()
print(f"  supplier-award rows pulled: {len(awards):,}")
print(f"  with a Companies House number on the notice: {awards['ch_number'].notna().sum():,}")
awards.head(3)

## 7. Match suppliers to the spine and write contract_win signals
Run each supplier through the ladder, keep the ones that matched a company in our dataset, and
write them into `signals`. We clear any previous Contracts Finder rows first so re-runs do not
double count.

In [ ]:
results = []
for row in awards.itertuples(index=False):
    cn, conf, method = match_supplier(row.supplier_name, row.ch_number)
    if cn:
        results.append((cn, row.award_date, row.value, row.detail, conf, method))

matched = pd.DataFrame(results, columns=["company_number", "award_date", "value", "detail",
                                         "confidence", "method"])
# tidy types: signal_date as a real date, value as a number
matched["signal_date"] = pd.to_datetime(matched["award_date"], errors="coerce").dt.date
matched["value"] = pd.to_numeric(matched["value"], errors="coerce")
matched["signal_type"] = "contract_win"
matched["source"] = "contracts_finder"
matched["retrieved_at"] = NOW

print(f"suppliers matched to a company in the dataset: {len(matched):,} of {len(awards):,}")
print("\nby method:")
print(matched["method"].value_counts().to_string())

con.execute("DELETE FROM signals WHERE source = 'contracts_finder'")
ins = matched[["company_number", "signal_type", "signal_date", "value", "detail",
               "source", "confidence", "retrieved_at"]]
con.register("tmp_sig", ins)
con.execute("""INSERT INTO signals
               SELECT company_number, signal_type, signal_date, value, detail,
                      source, confidence, retrieved_at
               FROM tmp_sig""")
con.unregister("tmp_sig")
print(f"\nwritten to signals: {len(ins):,} rows")

## 8. Summary and a look at the result
How many companies in the dataset now carry a contract-win signal, and a sample joined back to
their names. Then close the database so the file is flushed.

In [ ]:
n_companies = con.execute(
    "SELECT count(DISTINCT company_number) FROM signals WHERE source='contracts_finder'"
).fetchone()[0]
total = con.execute("SELECT count(*) FROM companies").fetchone()[0]
print(f"companies with a contract-win signal: {n_companies:,} of {total:,} ({n_companies/total:.2%})")

sample = con.execute("""
    SELECT c.company_name, c.sector, s.signal_date, s.value, s.confidence, s.detail
    FROM signals s JOIN companies c ON c.company_number = s.company_number
    WHERE s.source = 'contracts_finder'
    ORDER BY s.value DESC NULLS LAST
    LIMIT 10
""").df()
sample

In [ ]:
con.close()
print("saved:", DB_PATH)

## Notes and what comes next
- Coverage here depends on the spine you built. On a 50k sample only a handful of suppliers will
  fall inside it; run NB05 with `SAMPLE_N = None` first for a realistic number.
- Fuzzy matches (method `name_fuzzy`) and ambiguous exact matches (`name_exact_ambiguous`) carry
  lower confidence on purpose. You can tighten `FUZZY_CUTOFF` or add postcode disambiguation if
  you see false matches in the sample above.
- Next signal sources, same pattern (match by name, write to `signals` with confidence):
  Adzuna jobs (hiring as a growth signal, needs a free API key) and the harmonised news features
  from the unstructured-data-lab branch.
